# Hamkke (함께) on a free Colab GPU
Runtime → Change runtime type → **T4 GPU**. Run cells top to bottom. Each step writes to `runs/`; download results at the end.

In [ ]:
!git clone https://github.com/askarbekkk/hamkke-mt.git
%cd hamkke-mt
!pip install -q peft sacrebleu sentence-transformers

HyperCLOVA X SEED requires accepting its license on Hugging Face (instant). Open https://huggingface.co/naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B, click *Agree*, then paste a free read token from https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 1. Benchmark Korea's small open models (~20–30 min each)

In [ ]:
for m in [
    "kakaocorp/kanana-2-1.3b-instruct",
    "naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B",
    "LGAI-EXAONE/EXAONE-4.0-1.2B",
    "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"
]:
    !python finetune.py --eval-only --base {m} --langs all --n 50

## 2. Fine-tune Qwen2.5-1.5B on the weakest languages (~1–2 h)
Evaluates the base model, filters OPUS data with LaBSE, trains LoRA, evaluates again.

In [ ]:
!python finetune.py --langs khm_Khmr,npi_Deva,mya_Mymr --pairs 10000 --n 50

## 3. Results

In [ ]:
!python bench.py report --n 50
from IPython.display import Image
Image('results.png')

In [ ]:
!zip -qr hamkke_results.zip runs results.md results.csv results.png adapters
from google.colab import files
files.download('hamkke_results.zip')